<a href="https://colab.research.google.com/github/chegoorisaikiranmudiraj/data-pipeline/blob/main/sales_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np

In [8]:
from google.colab import auth
auth.authenticate_user()

In [9]:
from google.cloud import storage,bigquery

In [10]:
!gcloud projects list

PROJECT_ID         NAME        PROJECT_NUMBER  ENVIRONMENT
sales-data-506808  Sales data  379192895719


In [11]:
client = storage.Client(project='sales-data-506808')

In [12]:
buckets = client.list_buckets()

print("Buckets in your project:")
for bucket in buckets:
    print(f"- {bucket.name}")


Buckets in your project:
- sales-analysis-raw-data
- store-sales-raw-data


In [13]:
bucket = client.get_bucket("store-sales-raw-data")

In [14]:
# Creating new folder

blob = bucket.blob("sales/")
blob.upload_from_string("")

print("Created folder 'sales/' successfully")

Created folder 'sales/' successfully


In [15]:
# Uploading file to csv

blob = bucket.blob("sales/retail_sales.csv")
blob.upload_from_filename("/content/retail_store_sales.csv")

print("File uploaded sucessfully")

FileNotFoundError: [Errno 2] No such file or directory: '/content/retail_store_sales.csv'

In [16]:
# Reading the retail_sales file

blob = bucket.blob("sales/retail_sales.csv")
blob.download_to_filename("retail_sales.csv")

In [17]:
df = pd.read_csv("retail_sales.csv")

In [18]:
df

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
...,...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4.0,152.0,Credit Card,In-store,2023-09-03,NaN
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9.0,58.5,Cash,Online,2022-08-12,False
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10.0,140.0,Cash,Online,2024-08-24,NaN
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6.0,84.0,Cash,Online,2023-12-30,True


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [20]:
columns= ['Transaction_ID', 'Customer_ID', 'Category', 'Item', 'Price_Per_Unit',
       'Quantity', 'Total_Spent', 'Payment_Method', 'Location',
       'Transaction_Date', 'Discount_Applied']
columns = [c.lower() for c in columns]

In [21]:
df.columns = columns

In [22]:
df.isnull().sum()

,0
transaction_id,0
customer_id,0
category,0
item,1213
price_per_unit,609
quantity,604
total_spent,604
payment_method,0
location,0
transaction_date,0


In [23]:
#Finding discount amount

df["discount_amt"]=(df['price_per_unit']*df['quantity'])-df['total_spent']



In [24]:
#Fill missing discount_applied values

df.loc[df["discount_applied"].isna(), "discount_applied"] = (
    df.loc[df["discount_applied"].isna(), "discount_amt"] > 0
)

In [26]:
#Filling missing item values

item_map = df.dropna(subset=['item']).groupby('price_per_unit')['item'].first()
df['item'] = df['item'].fillna(df['price_per_unit'].map(item_map))

In [25]:
#Filling missing price_per_unit values


def fillingmissingvalues(df,target,formula):
  mask = df[target].isna()
  df.loc[mask, target] = formula(df.loc[mask])
  return df

fillingmissingvalues(df,"price_per_unit", lambda sub: sub['total_spent']/sub['quantity'])

,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,discount_amt
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,0.0
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,0.0
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,0.0
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,False,0.0
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4.0,152.0,Credit Card,In-store,2023-09-03,False,0.0
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9.0,58.5,Cash,Online,2022-08-12,False,0.0
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10.0,140.0,Cash,Online,2024-08-24,False,0.0
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6.0,84.0,Cash,Online,2023-12-30,True,0.0


In [27]:
#Filling missing quantity values

fillingmissingvalues(df,"quantity", lambda sub: sub['total_spent']/sub['price_per_unit'])

,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,discount_amt
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,0.0
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,0.0
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,0.0
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,False,0.0
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4.0,152.0,Credit Card,In-store,2023-09-03,False,0.0
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9.0,58.5,Cash,Online,2022-08-12,False,0.0
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10.0,140.0,Cash,Online,2024-08-24,False,0.0
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6.0,84.0,Cash,Online,2023-12-30,True,0.0


In [28]:
#Filling missing total values

fillingmissingvalues(df,"total_spent", lambda sub: sub['quantity']*sub['price_per_unit'])

,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,discount_amt
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,0.0
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,0.0
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,0.0
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,False,0.0
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4.0,152.0,Credit Card,In-store,2023-09-03,False,0.0
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9.0,58.5,Cash,Online,2022-08-12,False,0.0
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10.0,140.0,Cash,Online,2024-08-24,False,0.0
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6.0,84.0,Cash,Online,2023-12-30,True,0.0


In [32]:
df[df['quantity'].isna() & df['total_spent'].isna()]

,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,discount_amt


In [31]:
#Droping rows that has missing values in both quantity and total spent

df = df.dropna(subset=['quantity','total_spent'], how='all')

In [33]:
df.to_csv("retail_sales_cleaned.csv", index=False)

In [ ]:
blob = bucket.blob('sales/retail_sales_cleaned.csv')
blob.upload_from_filename('retail_sales_cleaned.csv')